# 1. Data Preprocessing Pipeline

## Overview
This notebook implements the **data cleaning and preprocessing pipeline** for the German EV Charging Station Registry (*Ladesäulenregister*).

### Data Source
The raw dataset is sourced from the **Bundesnetzagentur** (German Federal Network Agency), which maintains the official registry of all publicly accessible electric vehicle charging stations in Germany.

### Processing Steps
1. **Column translation**: Rename German column headers to English
2. **Data type conversion**: Fix numeric formats (comma → dot decimal), parse dates
3. **Charger type standardization**: Map German labels to English (`fast`/`normal`)
4. **Null value handling**: Fill missing plug/power data with defaults
5. **City name normalization**: Correct misspellings, merge district-level entries
6. **Deduplication**: Remove duplicate rows
7. **Export**: Save cleaned dataset for downstream analysis

In [ ]:
import pandas as pd
import numpy as np

## 1.1 Load Raw Data

The raw CSV uses Latin-1 encoding (German characters), semicolons as delimiters, and has 10 header rows to skip.

In [ ]:
# Load the raw German Charging Station Registry CSV
df = pd.read_csv('../data/raw/Ladesaeulenregister_CSV.csv', encoding='latin_1', sep=';', skiprows=10)
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
df.head(3)

## 1.2 Rename Columns to English

Translate all German column names to standardized English equivalents for international readability.

In [ ]:
# Mapping of original German column names to English equivalents
column_mapping = {
    'Betreiber': 'operator',
    'Straße': 'address',  # Note: may appear as 'Stra\xc3\x9fe' due to encoding
    'Hausnummer': 'house_number',
    'Adresszusatz': 'placeholder1',
    'Postleitzahl': 'postcode',
    'Ort': 'city',
    'Bundesland': 'federal_state',
    'Kreis/kreisfreie Stadt': 'metropolitan_area',
    'Breitengrad': 'latitude_[dg]',
    'Längengrad': 'longitude_[dg]',  # Note: may appear garbled due to encoding
    'Inbetriebnahmedatum': 'commissioning_date',
    'Anschlussleistung': 'power_connection_[kw]',
    'Normalladeeinrichtung': 'type_of_charger',
    'Anzahl Ladepunkte': 'number_of_charging_points',
    'Steckertypen1': 'type_of_plug_1',
    'P1 [kW]': 'p1_[kw]',
    'Public Key1': 'public_key1',
    'Steckertypen2': 'type_of_plug_2',
    'P2 [kW]': 'p2_[kw]',
    'Public Key2': 'public_key2',
    'Steckertypen3': 'type_of_plug_3',
    'P3 [kW]': 'p3_[kw]',
    'Public Key3': 'public_key3',
    'Steckertypen4': 'type_of_plug_4',
    'P4 [kW]': 'p4_[kw]',
    'Public Key4': 'public_key4'
}
df.rename(columns=column_mapping, inplace=True)

## 1.3 Standardize Charger Types

Map German charger type labels:
- `Schnellladeeinrichtung` → `fast` (DC fast charging, typically ≥50 kW)
- `Normalladeeinrichtung` → `normal` (AC charging, typically ≤22 kW)

In [ ]:
# Translate charger type from German to English
charger_mapping = {
    'Schnellladeeinrichtung': 'fast',
    'Normalladeeinrichtung': 'normal'
}
df.type_of_charger.replace(charger_mapping, inplace=True)
print('Charger type distribution:')
print(df['type_of_charger'].value_counts())

## 1.4 Handle Missing Values and Drop Unnecessary Columns

In [ ]:
# Fill null values with '0' for secondary plug/power columns (not all stations have 4 plugs)
na_columns = ['type_of_plug_2', 'p2_[kw]', 'type_of_plug_3', 'p3_[kw]', 'type_of_plug_4', 'p4_[kw]']
for column in na_columns:
    df[column] = df[column].fillna(value='0')

# Drop public key columns (not needed for spatial analysis)
df.drop(columns=['public_key1', 'public_key2', 'public_key3', 'public_key4'], inplace=True)

## 1.5 Fix Data Types

German numeric format uses commas as decimal separators. Convert to standard float format.

In [ ]:
# Convert German decimal format (comma) to standard float format (dot)
numeric_columns = ['longitude_[dg]', 'latitude_[dg]', 'power_connection_[kw]',
                   'p1_[kw]', 'p2_[kw]', 'p3_[kw]', 'p4_[kw]']
for column in numeric_columns:
    df[column] = df[column].str.replace(',', '.').astype(float)

# Parse commissioning date from German date format (DD.MM.YYYY)
df['commissioning_date'] = pd.to_datetime(df['commissioning_date'], format='%d.%m.%Y')

# Strip leading/trailing whitespace from all text columns
object_columns = df.select_dtypes(include='object').columns
for column in object_columns:
    df[column] = df[column].str.strip()

print(f'Date range: {df["commissioning_date"].min()} to {df["commissioning_date"].max()}')
df.dtypes

## 1.6 Normalize City Names

Fix encoding issues and merge district-level entries into their parent city.

In [ ]:
# Fix city names: merge district-level entries and correct encoding artifacts
city_modifications = {
    'M\u00fcnchen': 'München',           # Fix potential encoding issues
    'Frankfurt': 'Frankfurt am Main',    # Disambiguate from Frankfurt (Oder)
    'Frankfurt-Niederrad': 'Frankfurt am Main',
    'Stuttgart-Obertürkheim': 'Stuttgart',
    'Stuttgart-Mühlhausen': 'Stuttgart',
    'Stuttgart-Möhringen': 'Stuttgart',
}
df['city'].replace(city_modifications, inplace=True)

## 1.7 Remove Duplicates and Export

In [ ]:
# Remove exact duplicate rows
n_before = len(df)
df.drop_duplicates(inplace=True)
n_after = len(df)
print(f'Removed {n_before - n_after} duplicate rows ({n_before} → {n_after})')

# Save the cleaned DataFrame
df.to_csv('../data/processed/ChargingStationCleaned.csv', index=False)
print(f'Saved cleaned dataset: {len(df)} charging stations')
df.head()